# 00 — Setup & Raw Data Download

Downloads TLC Trip Record Parquet files from the official CloudFront CDN. Configure the years, months, and vehicle types below, then run all cells in order.

## Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.paths import ensure_dirs, iter_download_targets, str_path
from core.config.settings import settings

# ── Customize download scope here ────────────────────────────────────────────
YEARS         = [2024, 2025]
MONTHS        = list(range(1, 13))  # 1–12 for all months
VEHICLE_TYPES = ["yellow", "green"]  # Add "fhv", "hvfhv" as needed
SKIP_EXISTING = True  # Set False to re-download existing files
# ─────────────────────────────────────────────────────────────────────────────

ensure_dirs()
print(f"Project root : {settings.PROJECT_ROOT}")
print(f"TLC base URL : {settings.TLC_DATA_URL}")
print(f"Years        : {YEARS}")
print(f"Vehicle types: {VEHICLE_TYPES}")

## Download Zone Lookup Table

In [ ]:
import requests
from pathlib import Path
from src.paths import TAXI_ZONE_LOOKUP

LOOKUP_URL = f"{settings.TLC_LOOKUP_URL}/taxi_zone_lookup.csv"

if not TAXI_ZONE_LOOKUP.exists():
    print(f"Downloading zone lookup → {TAXI_ZONE_LOOKUP}")
    resp = requests.get(LOOKUP_URL, timeout=30)
    resp.raise_for_status()
    TAXI_ZONE_LOOKUP.parent.mkdir(parents=True, exist_ok=True)
    TAXI_ZONE_LOOKUP.write_bytes(resp.content)
    print(f"Saved {len(resp.content):,} bytes")
else:
    print(f"Zone lookup already present: {TAXI_ZONE_LOOKUP}")

## Download Trip Parquet Files

In [ ]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=30))
def download_file(url: str, dest: Path) -> int:
    """Download a URL to disk with retries. Returns size in bytes."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=settings.HTTP_TIMEOUT) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                fh.write(chunk)
    return dest.stat().st_size

targets = list(iter_download_targets(
    vehicle_types=VEHICLE_TYPES,
    years=YEARS,
    months=MONTHS,
    skip_existing=SKIP_EXISTING,
))

print(f"Files to download: {len(targets)}")

for i, (vt, year, month, url, path) in enumerate(targets, 1):
    print(f"[{i}/{len(targets)}] {vt} {year}-{month:02d} → {path.name} ...", end=" ")
    try:
        size = download_file(url, path)
        print(f"OK ({size / 1_048_576:.1f} MB)")
    except Exception as exc:
        print(f"FAILED: {exc}")

print("\nDownload complete.")

## Verify Downloaded Files

In [ ]:
from src.paths import RAW
import os

for vt in VEHICLE_TYPES:
    files = list(RAW[vt].glob("*.parquet"))
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    print(f"{vt:10s}: {len(files):3d} files | {total_mb:,.1f} MB")